In [2]:
import pandas as pd
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

input_csv = "/content/drive/MyDrive/EPdataset/NF-UQ-NIDS-v2.csv"

local_train = "/content/train.csv"
local_test = "/content/test.csv"

drive_output_folder = "/content/drive/MyDrive/EPdataset/stratified_splits"
os.makedirs(drive_output_folder, exist_ok=True)

drive_train = os.path.join(drive_output_folder, 'train.csv')
drive_test = os.path.join(drive_output_folder, 'test.csv')

columns_to_keep = [
    'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT',
    'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS',
    'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
    'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT',
    'L4_DST_PORT', 'ICMP_TYPE', 'DNS_QUERY_TYPE', 'DNS_QUERY_ID',
    'FTP_COMMAND_RET_CODE', 'Label', 'Attack'
]

exact_attack_counts = {
    "Benign": 25165295,
    "scanning": 3781419,
    "DDoS": 21748351,
    "Fuzzers": 22310,
    "Reconnaissance": 2633778,
    "Backdoor": 18978,
    "injection": 684897,
    "Bot": 143097,
    "DoS": 17875585,
    "Generic": 16560,
    "Brute Force": 123982,
    "Analysis": 2299,
    "password": 1153323,
    "Shellcode": 1427,
    "xss": 2455020,
    "mitm": 7723,
    "Infilteration": 116361,
    "Worms": 164,
    "Exploits": 31551,
    "ransomware": 3425,
    "Theft": 2431
}

train_quotas = {attack: round(count * 0.70) for attack, count in exact_attack_counts.items()}
train_counts_so_far = {attack: 0 for attack in exact_attack_counts.keys()}

# Write headers to local files
with open(local_train, 'w') as f_train, open(local_test, 'w') as f_test:
    f_train.write(','.join(columns_to_keep) + '\n')
    f_test.write(','.join(columns_to_keep) + '\n')

print("Starting stratified split (writing locally)...")

chunk_size = 1_000_000
chunk_iterator = pd.read_csv(input_csv, chunksize=chunk_size, low_memory=False)

total_train = 0
total_test = 0

for i, chunk in enumerate(chunk_iterator):

    # Drop unwanted columns
    chunk = chunk[columns_to_keep]

    train_rows = []
    test_rows = []

    for attack_type, group in chunk.groupby('Attack'):
        if attack_type not in train_quotas:
            print(f"WARNING: Unexpected label '{attack_type}' ({len(group)} rows) → test")
            test_rows.append(group)
            continue

        group_size = len(group)
        quota_remaining = train_quotas[attack_type] - train_counts_so_far[attack_type]

        if quota_remaining >= group_size:
            train_rows.append(group)
            train_counts_so_far[attack_type] += group_size
        elif quota_remaining > 0:
            train_rows.append(group.iloc[:quota_remaining])
            test_rows.append(group.iloc[quota_remaining:])
            train_counts_so_far[attack_type] += quota_remaining
        else:
            test_rows.append(group)

    if train_rows:
        batch = pd.concat(train_rows)
        batch.to_csv(local_train, mode='a', header=False, index=False)
        total_train += len(batch)

    if test_rows:
        batch = pd.concat(test_rows)
        batch.to_csv(local_test, mode='a', header=False, index=False)
        total_test += len(batch)

    print(f"Chunk {i+1} done | Train: {total_train:,} | Test: {total_test:,} | Total: {total_train+total_test:,}")

print(f"\n=== SPLIT COMPLETE ===")
print(f"Train rows : {total_train:,}")
print(f"Test rows  : {total_test:,}")
print(f"Total rows : {total_train + total_test:,}")
print(f"Expected   : {sum(exact_attack_counts.values()):,}")
print(f"Difference : {sum(exact_attack_counts.values()) - (total_train + total_test):,}")

print(f"\n=== TRAIN QUOTA FILL ===")
for attack, quota in train_quotas.items():
    filled = train_counts_so_far[attack]
    print(f"  {attack:20s}: {filled:>10,} / {quota:>10,}")

# Push to Drive once at the end
print("\nCopying to Drive...")
shutil.copy(local_train, drive_train)
shutil.copy(local_test, drive_test)
print("Done! Files pushed to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting stratified split (writing locally)...
Chunk 1 done | Train: 1,000,000 | Test: 0 | Total: 1,000,000
Chunk 2 done | Train: 2,000,000 | Test: 0 | Total: 2,000,000
Chunk 3 done | Train: 3,000,000 | Test: 0 | Total: 3,000,000
Chunk 4 done | Train: 4,000,000 | Test: 0 | Total: 4,000,000
Chunk 5 done | Train: 5,000,000 | Test: 0 | Total: 5,000,000
Chunk 6 done | Train: 6,000,000 | Test: 0 | Total: 6,000,000
Chunk 7 done | Train: 7,000,000 | Test: 0 | Total: 7,000,000
Chunk 8 done | Train: 8,000,000 | Test: 0 | Total: 8,000,000
Chunk 9 done | Train: 9,000,000 | Test: 0 | Total: 9,000,000
Chunk 10 done | Train: 10,000,000 | Test: 0 | Total: 10,000,000
Chunk 11 done | Train: 11,000,000 | Test: 0 | Total: 11,000,000
Chunk 12 done | Train: 12,000,000 | Test: 0 | Total: 12,000,000
Chunk 13 done | Train: 13,000,000 | Test: 0 | Total: 13,000,000
Chunk 14 done | Tra

In [ ]:
#Dataset Checking


import pandas as pd

input_csv = "/content/drive/MyDrive/EPdataset/NF-UQ-NIDS-v2.csv"

# Read just the first chunk
chunk = pd.read_csv(input_csv, nrows=100000, low_memory=False)

print("=== COLUMNS ===")
print(chunk.columns.tolist())

print("\n=== DTYPES ===")
print(chunk.dtypes)

print("\n=== NULL COUNTS ===")
print(chunk.isnull().sum())

print("\n=== UNIQUE ATTACK LABELS ===")
print(chunk['Attack'].unique())

print("\n=== SAMPLE ROWS ===")
print(chunk.head(3))